In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.gold")

print("✓ Gold layer environment initialized")
print(f"✓ Transformation timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
from pyspark.sql import functions as F

# Transform customers with advanced segmentation and metrics
df_customers = spark.table("my_catalog.silver.customers")

# Customer segmentation logic
df_customers_gold = df_customers.select(
    F.col("Customer_Id"),
    F.col("First_Name"),
    F.col("Last_Name"),
    F.concat_ws(" ", F.col("First_Name"), F.col("Last_Name")).alias("Full_Name"),
    F.col("Email"),
    F.col("Company"),
    F.col("City"),
    F.col("Country"),
    F.col("Phone_1"),
    F.col("Phone_2"),
    F.col("Subscription_Date"),
    
    # Derive customer tenure
    F.datediff(F.current_date(), F.col("Subscription_Date")).alias("Customer_Tenure_Days"),
    
    # Customer segmentation by tenure
    F.when(F.datediff(F.current_date(), F.col("Subscription_Date")) < 180, "New")
     .when(F.datediff(F.current_date(), F.col("Subscription_Date")) < 730, "Active")
     .otherwise("Loyal").alias("Customer_Segment"),
    
    # Email domain for B2B analysis
    F.regexp_extract(F.col("Email"), "@(.+)$", 1).alias("Email_Domain"),
    
    # Geographic region
    F.when(F.col("Country").isin(["United States", "Canada", "Mexico"]), "North America")
     .when(F.col("Country").isin(["United Kingdom", "Germany", "France", "Spain", "Italy"]), "Europe")
     .when(F.col("Country").isin(["China", "Japan", "India", "Singapore"]), "Asia Pacific")
     .otherwise("Other").alias("Region"),
    
    # Audit column
    F.current_timestamp().alias("gold_processed_timestamp")
)

# Write to gold layer
df_customers_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.gold.customers_gold")

# Summary statistics
total_customers = df_customers_gold.count()
segment_distribution = df_customers_gold.groupBy("Customer_Segment").count().collect()
region_distribution = df_customers_gold.groupBy("Region").count().collect()

print("✓ Customers Gold table created")
print(f"  - Total customers: {total_customers:,}")
print("\n  Customer Segments:")
for row in segment_distribution:
    pct = (row['count'] / total_customers) * 100
    print(f"    - {row['Customer_Segment']}: {row['count']:,} ({pct:.1f}%)")
print("\n  Regional Distribution:")
for row in region_distribution:
    pct = (row['count'] / total_customers) * 100
    print(f"    - {row['Region']}: {row['count']:,} ({pct:.1f}%)")

In [0]:
from pyspark.sql import functions as F

# Advanced lead analytics with conversion metrics
df_leads = spark.table("my_catalog.silver.leads")

df_lead_metrics = df_leads.select(
    F.col("Account_Id"),
    F.col("Company"),
    F.col("Full_Name").alias("Contact_Name"),
    F.col("First_Name"),
    F.col("Last_Name"),
    F.col("Primary_Email").alias("Email"),
    F.col("Phone_1").alias("Phone"),
    F.col("Website"),
    F.col("Source"),
    F.col("Deal_Stage"),
    F.col("Deal_Stage_Category"),
    
    # Lead quality scoring based on deal stage
    F.when(F.col("Deal_Stage_Category") == "Converted", 100)
     .when(F.col("Deal_Stage_Category") == "Active", 75)
     .when(F.col("Deal_Stage_Category") == "New", 50)
     .otherwise(25).alias("Lead_Quality_Score"),
    
    # Status classification
    F.when(F.col("Deal_Stage_Category") == "Active", 1).otherwise(0).alias("Is_Active"),
    F.when(F.col("Deal_Stage_Category") == "Converted", 1).otherwise(0).alias("Is_Converted"),
    
    # Contact information flags
    F.col("Has_Multiple_Contacts"),
    F.when(F.col("Email_2").isNotNull(), 1).otherwise(0).alias("Has_Secondary_Email"),
    F.when(F.col("Phone_2").isNotNull(), 1).otherwise(0).alias("Has_Secondary_Phone"),
    
    # Audit column
    F.current_timestamp().alias("gold_processed_timestamp")
)

# Write to gold layer
df_lead_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.gold.lead_conversion_metrics")

# Calculate conversion metrics
total_leads = df_lead_metrics.count()
active_leads = df_lead_metrics.filter(F.col("Is_Active") == 1).count()
converted_leads = df_lead_metrics.filter(F.col("Is_Converted") == 1).count()
conversion_rate = (converted_leads / total_leads) * 100 if total_leads > 0 else 0

stage_distribution = df_lead_metrics.groupBy("Deal_Stage_Category").count().orderBy(F.desc("count")).collect()
source_distribution = df_lead_metrics.groupBy("Source").count().orderBy(F.desc("count")).collect()

print("✓ Lead Conversion Metrics Gold table created")
print(f"  - Total leads: {total_leads:,}")
print(f"  - Active leads: {active_leads:,}")
print(f"  - Converted leads: {converted_leads:,}")
print(f"  - Conversion rate: {conversion_rate:.2f}%")
print("\n  Deal Stage Distribution:")
for row in stage_distribution:
    pct = (row['count'] / total_leads) * 100
    print(f"    - {row['Deal_Stage_Category']}: {row['count']:,} ({pct:.1f}%)")
print("\n  Top Lead Sources:")
for row in source_distribution[:5]:
    pct = (row['count'] / total_leads) * 100
    print(f"    - {row['Source']}: {row['count']:,} ({pct:.1f}%)")

In [0]:
from pyspark.sql import functions as F

# Product analytics with pricing and availability insights
df_products = spark.table("my_catalog.silver.products")

df_product_performance = df_products.select(
    F.col("Internal_ID").alias("Product_Id"),
    F.col("Name").alias("Product_Name"),
    F.col("Description"),
    F.col("Brand"),
    F.col("Category"),
    F.col("Price"),
    F.col("Currency"),
    F.col("Stock"),
    F.col("Stock_Status"),
    F.col("Availability"),
    F.col("Is_Available"),
    F.col("Color"),
    F.col("Size"),
    F.col("EAN"),
    
    # Price tier categorization
    F.when(F.col("Price") > 800, "Premium")
     .when(F.col("Price") > 400, "Mid-Range")
     .otherwise("Budget").alias("Price_Tier"),
    
    # Inventory status categorization
    F.when(F.col("Is_Available") == True, "Available")
     .when((F.col("Stock") > 0) & (F.col("Is_Available") == False), "In Stock - Unavailable")
     .otherwise("Out of Stock").alias("Inventory_Status_Category"),
    
    # Price ranges for analytics
    F.floor(F.col("Price") / 100).cast("integer").alias("Price_Range_Bucket"),
    
    # Product options flags
    F.col("Has_Size_Options"),
    F.col("Has_Color_Options"),
    
    # Audit column
    F.current_timestamp().alias("gold_processed_timestamp")
)

# Write to gold layer
df_product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.gold.product_performance")

# Calculate product metrics
total_products = df_product_performance.count()
avg_price = df_product_performance.select(F.avg("Price")).collect()[0][0]
min_price = df_product_performance.select(F.min("Price")).collect()[0][0]
max_price = df_product_performance.select(F.max("Price")).collect()[0][0]
available_products = df_product_performance.filter(F.col("Is_Available") == True).count()

tier_distribution = df_product_performance.groupBy("Price_Tier").count().orderBy(F.desc("count")).collect()
category_distribution = df_product_performance.groupBy("Category").agg(
    F.count("*").alias("Product_Count"),
    F.avg("Price").alias("Avg_Price")
).orderBy(F.desc("Product_Count")).collect()
brand_distribution = df_product_performance.groupBy("Brand").count().orderBy(F.desc("count")).collect()

print("✓ Product Performance Gold table created")
print(f"  - Total products: {total_products:,}")
print(f"  - Available products: {available_products:,}")
print(f"  - Average price: ${avg_price:.2f}")
print(f"  - Price range: ${min_price:.2f} - ${max_price:.2f}")
print("\n  Price Tier Distribution:")
for row in tier_distribution:
    pct = (row['count'] / total_products) * 100
    print(f"    - {row['Price_Tier']}: {row['count']:,} ({pct:.1f}%)")
print("\n  Top Categories by Product Count:")
for row in category_distribution[:5]:
    print(f"    - {row['Category']}: {row['Product_Count']} products (Avg: ${row['Avg_Price']:.2f})")
print("\n  Top Brands:")
for row in brand_distribution[:5]:
    pct = (row['count'] / total_products) * 100
    print(f"    - {row['Brand']}: {row['count']:,} ({pct:.1f}%)")

In [0]:
from pyspark.sql import functions as F

# Organization analytics with industry and size segmentation
df_organizations = spark.table("my_catalog.silver.organizations")

df_org_intelligence = df_organizations.select(
    F.col("Organization_Id"),
    F.col("Name").alias("Organization_Name"),
    F.col("Website"),
    F.col("Country"),
    F.col("Description"),
    F.col("Founded"),
    F.col("Industry"),
    F.col("Number_of_employees").alias("Number_of_Employees"),
    
    # Use existing calculated columns from silver layer
    F.col("Company_Age"),
    F.col("Company_Age_Group").alias("Company_Maturity"),
    F.col("Company_Size").alias("Organization_Size"),
    
    # Geographic region
    F.when(F.col("Country").isin(["United States", "Canada", "Mexico"]), "North America")
     .when(F.col("Country").isin(["United Kingdom", "Germany", "France", "Spain", "Italy"]), "Europe")
     .when(F.col("Country").isin(["China", "Japan", "India", "Singapore"]), "Asia Pacific")
     .otherwise("Other").alias("Region"),
    
    # Website domain
    F.regexp_extract(F.col("Website"), "^https?://(?:www\\.)?(.*?)(?:/|$)", 1).alias("Domain"),
    
    # Additional flags
    F.col("Has_Website"),
    F.col("Is_Recent_Founded"),
    
    # Audit column
    F.current_timestamp().alias("gold_processed_timestamp")
)

# Write to gold layer
df_org_intelligence.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.gold.organization_intelligence")

# Calculate organization metrics
total_orgs = df_org_intelligence.count()
avg_employees = df_org_intelligence.select(F.avg("Number_of_Employees")).collect()[0][0]
avg_age = df_org_intelligence.select(F.avg("Company_Age")).collect()[0][0]

size_distribution = df_org_intelligence.groupBy("Organization_Size").count().orderBy(F.desc("count")).collect()
maturity_distribution = df_org_intelligence.groupBy("Company_Maturity").count().orderBy(F.desc("count")).collect()
industry_distribution = df_org_intelligence.groupBy("Industry").agg(
    F.count("*").alias("Org_Count"),
    F.avg("Number_of_Employees").alias("Avg_Employees")
).orderBy(F.desc("Org_Count")).collect()
region_distribution = df_org_intelligence.groupBy("Region").count().orderBy(F.desc("count")).collect()

print("✓ Organization Intelligence Gold table created")
print(f"  - Total organizations: {total_orgs:,}")
print(f"  - Average employees: {avg_employees:,.0f}")
print(f"  - Average company age: {avg_age:.1f} years")
print("\n  Organization Size Distribution:")
for row in size_distribution:
    pct = (row['count'] / total_orgs) * 100
    print(f"    - {row['Organization_Size']}: {row['count']:,} ({pct:.1f}%)")
print("\n  Company Maturity:")
for row in maturity_distribution:
    pct = (row['count'] / total_orgs) * 100
    print(f"    - {row['Company_Maturity']}: {row['count']:,} ({pct:.1f}%)")
print("\n  Regional Distribution:")
for row in region_distribution:
    pct = (row['count'] / total_orgs) * 100
    print(f"    - {row['Region']}: {row['count']:,} ({pct:.1f}%)")
print("\n  Top Industries:")
for row in industry_distribution[:5]:
    print(f"    - {row['Industry']}: {row['Org_Count']:,} orgs (Avg employees: {row['Avg_Employees']:.0f})")

In [0]:
from pyspark.sql import functions as F

# Join customers with organizations for complete business intelligence
df_customers = spark.table("my_catalog.gold.customers_gold")
df_organizations = spark.table("my_catalog.gold.organization_intelligence")

# Join based on company name matching
df_customer_org_360 = df_customers.join(
    df_organizations,
    df_customers.Company == df_organizations.Organization_Name,
    "left"
).select(
    # Customer information
    F.col("Customer_Id"),
    df_customers["Full_Name"].alias("Customer_Name"),
    df_customers["Email"],
    df_customers["Company"],
    df_customers["City"].alias("Customer_City"),
    df_customers["Country"].alias("Customer_Country"),
    df_customers["Customer_Segment"],
    df_customers["Customer_Tenure_Days"],
    df_customers["Region"].alias("Customer_Region"),
    
    # Organization information
    F.col("Organization_Id"),
    F.col("Organization_Name"),
    F.col("Industry"),
    F.col("Number_of_Employees"),
    F.col("Company_Age"),
    F.col("Company_Maturity"),
    F.col("Organization_Size"),
    F.col("Website"),
    df_organizations["Country"].alias("Organization_Country"),
    
    # Relationship indicators
    F.when(F.col("Organization_Id").isNotNull(), "Matched").otherwise("Unmatched").alias("Organization_Match_Status"),
    
    # Audit
    F.current_timestamp().alias("gold_processed_timestamp")
)

# Write to gold layer
df_customer_org_360.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.gold.customer_organization_360")

# Calculate match statistics
total_customers = df_customer_org_360.count()
matched_customers = df_customer_org_360.filter(F.col("Organization_Match_Status") == "Matched").count()
match_rate = (matched_customers / total_customers) * 100 if total_customers > 0 else 0

print("✓ Customer-Organization 360 Gold table created")
print(f"  - Total customer records: {total_customers:,}")
print(f"  - Matched to organizations: {matched_customers:,} ({match_rate:.1f}%)")
print(f"  - Unmatched customers: {total_customers - matched_customers:,}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime

# Executive dashboard with key business metrics

# Customer metrics
df_customers = spark.table("my_catalog.gold.customers_gold")
customer_metrics = df_customers.agg(
    F.count("*").alias("total_customers"),
    F.countDistinct("Country").alias("countries_served"),
    F.sum(F.when(F.col("Customer_Segment") == "Loyal", 1).otherwise(0)).alias("loyal_customers")
).collect()[0]

# Lead metrics
df_leads = spark.table("my_catalog.gold.lead_conversion_metrics")
lead_metrics = df_leads.agg(
    F.count("*").alias("total_leads"),
    F.sum("Is_Active").alias("active_leads"),
    F.sum("Is_Converted").alias("converted_leads"),
    F.avg("Lead_Quality_Score").alias("avg_lead_score")
).collect()[0]

conversion_rate = (lead_metrics['converted_leads'] / lead_metrics['total_leads']) * 100 if lead_metrics['total_leads'] > 0 else 0

# Product metrics
df_products = spark.table("my_catalog.gold.product_performance")
product_metrics = df_products.agg(
    F.count("*").alias("total_products"),
    F.avg("Price").alias("avg_product_price"),
    F.countDistinct("Category").alias("product_categories")
).collect()[0]

# Organization metrics
df_orgs = spark.table("my_catalog.gold.organization_intelligence")
org_metrics = df_orgs.agg(
    F.count("*").alias("total_organizations"),
    F.avg("Number_of_Employees").alias("avg_org_size"),
    F.countDistinct("Industry").alias("industries_covered")
).collect()[0]

# Create summary table
summary_data = [
    ("Customer Metrics", "Total Customers", customer_metrics['total_customers'], ""),
    ("Customer Metrics", "Countries Served", customer_metrics['countries_served'], ""),
    ("Customer Metrics", "Loyal Customers", customer_metrics['loyal_customers'], ""),
    ("Lead Metrics", "Total Leads", lead_metrics['total_leads'], ""),
    ("Lead Metrics", "Active Leads", lead_metrics['active_leads'], ""),
    ("Lead Metrics", "Converted Leads", lead_metrics['converted_leads'], ""),
    ("Lead Metrics", "Conversion Rate", conversion_rate, "%"),
    ("Lead Metrics", "Avg Lead Score", round(lead_metrics['avg_lead_score'], 2), ""),
    ("Product Metrics", "Total Products", product_metrics['total_products'], ""),
    ("Product Metrics", "Avg Product Price", round(product_metrics['avg_product_price'], 2), "$"),
    ("Product Metrics", "Product Categories", product_metrics['product_categories'], ""),
    ("Organization Metrics", "Total Organizations", org_metrics['total_organizations'], ""),
    ("Organization Metrics", "Avg Organization Size", round(org_metrics['avg_org_size'], 0), "employees"),
    ("Organization Metrics", "Industries Covered", org_metrics['industries_covered'], "")
]

schema = StructType([
    StructField("Category", StringType(), True),
    StructField("Metric_Name", StringType(), True),
    StructField("Metric_Value", DoubleType(), True),
    StructField("Unit", StringType(), True)
])

df_summary = spark.createDataFrame(summary_data, schema)

df_summary_enhanced = df_summary.withColumn(
    "Formatted_Value",
    F.when(F.col("Unit") == "$", F.concat(F.lit("$"), F.format_number(F.col("Metric_Value"), 2)))
     .when(F.col("Unit") == "%", F.concat(F.format_number(F.col("Metric_Value"), 2), F.lit("%")))
     .when(F.col("Unit") == "employees", F.concat(F.format_number(F.col("Metric_Value"), 0), F.lit(" employees")))
     .otherwise(F.format_number(F.col("Metric_Value"), 0))
).withColumn(
    "gold_processed_timestamp",
    F.current_timestamp()
)

# Write to gold layer
df_summary_enhanced.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_catalog.gold.business_summary_dashboard")

print("="*70)
print("✓ BUSINESS SUMMARY DASHBOARD - GOLD LAYER")
print("="*70)
print("\n📊 CUSTOMER METRICS")
print(f"  Total Customers: {customer_metrics['total_customers']:,}")
print(f"  Countries Served: {customer_metrics['countries_served']}")
print(f"  Loyal Customers: {customer_metrics['loyal_customers']:,}")

print("\n🎯 LEAD METRICS")
print(f"  Total Leads: {lead_metrics['total_leads']:,}")
print(f"  Active Leads: {lead_metrics['active_leads']:,}")
print(f"  Converted Leads: {lead_metrics['converted_leads']:,}")
print(f"  Conversion Rate: {conversion_rate:.2f}%")
print(f"  Avg Lead Quality Score: {lead_metrics['avg_lead_score']:.1f}")

print("\n📦 PRODUCT METRICS")
print(f"  Total Products: {product_metrics['total_products']:,}")
print(f"  Avg Product Price: ${product_metrics['avg_product_price']:.2f}")
print(f"  Product Categories: {product_metrics['product_categories']}")

print("\n🏢 ORGANIZATION METRICS")
print(f"  Total Organizations: {org_metrics['total_organizations']:,}")
print(f"  Avg Organization Size: {org_metrics['avg_org_size']:,.0f} employees")
print(f"  Industries Covered: {org_metrics['industries_covered']}")
print("\n" + "="*70)

In [0]:
# Validate all gold layer tables
print("="*70)
print("GOLD LAYER TRANSFORMATION VALIDATION")
print("="*70)

tables = spark.sql("SHOW TABLES IN my_catalog.gold").collect()

print(f"\nTotal Gold Tables Created: {len(tables)}\n")

table_stats = []

for table in tables:
    table_name = table['tableName']
    try:
        count = spark.table(f"my_catalog.gold.{table_name}").count()
        table_stats.append((table_name, count))
        print(f"✓ {table_name:<40} : {count:>15,} records")
    except Exception as e:
        print(f"✗ {table_name:<40} : Error - {str(e)}")

print("\n" + "="*70)
print("GOLD LAYER PIPELINE COMPLETED SUCCESSFULLY")
print("="*70)
print(f"\nTimestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nGold layer tables are ready for analytics and BI consumption.")